# Synthetic Co-60 Gamma Spectrum Example

This notebook demonstrates how to use `gs_creator.py` to generate a synthetic
gamma spectrum for **Cobalt-60 (Co-60)** and plot the result.

Co-60 decays via beta decay to Ni-60 and emits two prominent gamma rays:

| Energy (keV) | Emission probability |
|---|---|
| 1173.2 | 99.85 % |
| 1332.5 | 99.98 % |

## 1. Setup – import modules

In [ ]:
import sys
import os

# Add the package root to the path when running from the examples/ directory
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

import gs_creator
import gs_analysis

## 2. Define Co-60 source parameters

The two characteristic gamma-ray lines of Co-60 and their relative emission rates
(arbitrary units – scale as needed for your activity level).

In [ ]:
# Co-60 gamma energies in keV
co60_energies = [1173.2, 1332.5]

# Relative emission rates (proportional to emission probabilities)
# Scale the value to represent the number of detected counts at the photopeak
co60_emission_rates = [9985, 9998]  # ~1e4 counts at each peak

print("Co-60 gamma lines:")
for e, r in zip(co60_energies, co60_emission_rates):
    print(f"  {e:.1f} keV  –  emission rate: {r}")

## 3. Generate the synthetic spectrum

In [ ]:
# Spectrum settings
NUM_BINS = 4096
ENERGY_RANGE = (0.0, 2000.0)  # keV – covers both Co-60 lines
FWHM_FACTOR = 0.02            # 2 % energy resolution (typical NaI(Tl) scintillator)

co60_spectrum = gs_creator.create_spectrum_from_peaks(
    peak_energies=co60_energies,
    emission_rates=co60_emission_rates,
    num_bins=NUM_BINS,
    energy_range=ENERGY_RANGE,
    fwhm_factor=FWHM_FACTOR,
    include_compton=True,   # add Compton continuum for realism
    compton_fraction=0.4,
    spec_name="Co-60 synthetic spectrum",
)

print(f"Spectrum name    : {co60_spectrum.spec_name}")
print(f"Number of bins   : {co60_spectrum.num_channels}")
print(f"Total counts     : {co60_spectrum.counts.sum():,}")

## 4. Plot the spectrum

In [ ]:
# Build energy axis from the spectrum's energy-fit coefficients
ebins = gs_analysis.generate_ebins(co60_spectrum)  # keV per channel

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(ebins, co60_spectrum.counts, color='steelblue', linewidth=0.8,
        label='Co-60 synthetic spectrum')

# Annotate the two Co-60 photopeaks
for energy, label in zip(co60_energies, ['1173.2 keV', '1332.5 keV']):
    ax.axvline(x=energy, color='crimson', linestyle='--', linewidth=1.0, alpha=0.8)
    ax.text(energy + 10, ax.get_ylim()[1] * 0.85, label,
            color='crimson', fontsize=9, rotation=90, va='top')

ax.set_xlabel('Energy (keV)', fontsize=12)
ax.set_ylabel('Counts', fontsize=12)
ax.set_title('Synthetic Co-60 Gamma Spectrum', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(ENERGY_RANGE)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

## 5. Zoom in on the two photopeaks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

zoom_windows = [
    (1100.0, 1250.0, '1173.2 keV photopeak'),
    (1260.0, 1400.0, '1332.5 keV photopeak'),
]

for ax, (e_lo, e_hi, title) in zip(axes, zoom_windows):
    mask = (ebins >= e_lo) & (ebins <= e_hi)
    ax.plot(ebins[mask], co60_spectrum.counts[mask],
            color='steelblue', linewidth=1.2)
    ax.set_xlabel('Energy (keV)', fontsize=11)
    ax.set_ylabel('Counts', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(e_lo, e_hi)
    ax.set_ylim(bottom=0)

plt.suptitle('Co-60 Photopeaks (zoomed)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Optional – add a flat background

Use `create_flat_background` to simulate ambient radiation and re-generate the
spectrum on top of it.

In [ ]:
background = gs_creator.create_flat_background(
    num_bins=NUM_BINS,
    background_level=50,  # counts per bin
    energy_range=ENERGY_RANGE,
    spec_name="flat background",
)

co60_with_bg = gs_creator.create_spectrum_from_peaks(
    peak_energies=co60_energies,
    emission_rates=co60_emission_rates,
    num_bins=NUM_BINS,
    energy_range=ENERGY_RANGE,
    background_spectrum=background,
    fwhm_factor=FWHM_FACTOR,
    include_compton=True,
    compton_fraction=0.4,
    spec_name="Co-60 + background",
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ebins, co60_with_bg.counts, color='darkorange', linewidth=0.8,
        label='Co-60 + flat background (50 cts/bin)')
ax.plot(ebins, co60_spectrum.counts, color='steelblue', linewidth=0.8,
        linestyle='--', label='Co-60 (no background)', alpha=0.7)

for energy in co60_energies:
    ax.axvline(x=energy, color='crimson', linestyle=':', linewidth=0.9, alpha=0.7)

ax.set_xlabel('Energy (keV)', fontsize=12)
ax.set_ylabel('Counts', fontsize=12)
ax.set_title('Co-60 Spectrum With and Without Background', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(ENERGY_RANGE)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()